[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ksankaran/hello-model/blob/main/pytorch_models.ipynb)

# Hello, PyTorch!

In [Part 1](https://medium.com/@v31u/hello-model-build-your-first-ai-model-from-scratch-552ecdffdfe9), we built a model from scratch: `y = mx + b`, two knobs, gradient descent by hand.

In [Part 2](https://medium.com/@v31u/from-line-to-network-build-your-first-neural-network-from-scratch-b8d84d6708e0), we built a neural network: 10 knobs, ReLU bends, backpropagation by hand.

Both times, we wrote every gradient ourselves. In this notebook, we rebuild both models in PyTorch and let **autograd** handle the gradients automatically.

Same models. Same results. A fraction of the code.

---
## Part 1: Tensors and Autograd

PyTorch tensors are like Python lists, but they track every operation performed on them. This lets PyTorch compute gradients automatically.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Same data from Part 1
sqft  = [600, 800, 1000, 1200, 1500, 1800, 2200]
price = [150, 200,  250,  280,  350,  400,  500]

# As tensors
X_list = torch.tensor(sqft, dtype=torch.float32)
y_list = torch.tensor(price, dtype=torch.float32)

print(f"X tensor: {X_list}")
print(f"y tensor: {y_list}")
print(f"\nThey look like lists. But watch what happens next.")

In [ ]:
# Create parameters with requires_grad=True
# This tells PyTorch: "track every operation on these numbers"
m = torch.tensor(0.05, requires_grad=True)
b = torch.tensor(10.0, requires_grad=True)

# Forward pass - PyTorch records the computation graph
predictions = m * X_list + b
loss = ((predictions - y_list) ** 2).mean()

print(f"Predictions: {predictions.data}")
print(f"Loss: {loss.item():,.1f}")

# Now the magic: compute ALL gradients in one call
loss.backward()

print(f"\nGradient for m: {m.grad.item():,.1f}")
print(f"Gradient for b: {b.grad.item():.1f}")
print(f"\nPyTorch walked the computation graph backward and computed")
print(f"both gradients automatically. No compute_gradients() needed.")

### Compare: our manual gradients from Part 1

```python
# Part 1: 9 lines of gradient math
def compute_gradients(m, b):
    grad_m, grad_b = 0, 0
    n = len(sqft)
    for x, actual in zip(sqft, price):
        error = (m * x + b) - actual
        grad_m += (2/n) * error * x
        grad_b += (2/n) * error
    return grad_m, grad_b
```

PyTorch version: `loss.backward()`. One line. Same math.

In [ ]:
# Let's verify: manual gradients match PyTorch gradients

def compute_gradients_manual(m_val, b_val):
    """Same gradient function from Part 1."""
    grad_m, grad_b = 0, 0
    n = len(sqft)
    for x, actual in zip(sqft, price):
        error = (m_val * x + b_val) - actual
        grad_m += (2/n) * error * x
        grad_b += (2/n) * error
    return grad_m, grad_b

manual_gm, manual_gb = compute_gradients_manual(0.05, 10.0)

print(f"Manual gradient for m: {manual_gm:,.1f}")
print(f"PyTorch gradient for m: {m.grad.item():,.1f}")
print(f"\nManual gradient for b: {manual_gb:.1f}")
print(f"PyTorch gradient for b: {b.grad.item():.1f}")
print(f"\nIdentical. Same math, zero bookkeeping.")

---
## Part 2: Linear Regression - Manual vs PyTorch

Let's rebuild the Part 1 model. First the manual version for comparison, then the PyTorch version.

In [ ]:
# --- Manual training (from Part 1) ---

def compute_loss_manual(m, b):
    total = 0
    for x, actual in zip(sqft, price):
        error = (m * x + b) - actual
        total += error ** 2
    return total / len(sqft)

m_manual = 0.05
b_manual = 10.0
lr = 0.0000003

for epoch in range(500):
    gm, gb = compute_gradients_manual(m_manual, b_manual)
    m_manual -= lr * gm
    b_manual -= lr * gb

print("--- Manual (Part 1) ---")
print(f"m = {m_manual:.4f}, b = {b_manual:.2f}")
print(f"Loss: {compute_loss_manual(m_manual, b_manual):.1f}")
print()
print("Sqft  | Actual | Predicted")
print("------|--------|----------")
for x, actual in zip(sqft, price):
    pred = m_manual * x + b_manual
    print(f"{x:5d} | ${actual:5d}k | ${pred:8.1f}k")

In [ ]:
# --- PyTorch version ---

# Data as 2D tensors (each sample is a row)
X = torch.tensor([[600],[800],[1000],[1200],[1500],[1800],[2200]],
                 dtype=torch.float32)
y = torch.tensor([[150],[200],[250],[280],[350],[400],[500]],
                 dtype=torch.float32)

# The model: nn.Linear(1, 1) = one input, one output = y = weight*x + bias
torch.manual_seed(42)
model = nn.Linear(1, 1)

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.0000003)

# Training loop
for epoch in range(500):
    predictions = model(X)
    loss = criterion(predictions, y)

    optimizer.zero_grad()   # reset gradients from last step
    loss.backward()         # compute gradients (autograd!)
    optimizer.step()        # update the knobs

    if epoch % 100 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():10.1f}")

# Extract learned parameters
w = model.weight.item()
bias = model.bias.item()
print(f"\n--- PyTorch ---")
print(f"weight = {w:.4f}, bias = {bias:.2f}")
print(f"Loss: {loss.item():.1f}")

In [ ]:
# Compare predictions side by side
print("Sqft  | Actual | Manual   | PyTorch")
print("------|--------|----------|--------")
for x_val, actual in zip(sqft, price):
    pred_manual = m_manual * x_val + b_manual
    pred_torch = w * x_val + bias
    print(f"{x_val:5d} | ${actual:5d}k | ${pred_manual:6.1f}k | ${pred_torch:6.1f}k")

print(f"\nBoth models learned essentially the same line.")
print(f"Manual: y = {m_manual:.4f}x + {b_manual:.2f}")
print(f"PyTorch: y = {w:.4f}x + {bias:.2f}")

### The training loops side by side

**Manual (Part 1):**
```python
for epoch in range(500):
    loss = compute_loss(m, b)
    grad_m, grad_b = compute_gradients(m, b)
    m = m - learning_rate * grad_m
    b = b - learning_rate * grad_b
```

**PyTorch:**
```python
for epoch in range(500):
    predictions = model(X)
    loss = criterion(predictions, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

Same structure: predict, measure, adjust, repeat. But we didn't write `compute_gradients()` or `compute_loss()`. PyTorch handles both.

---
## Part 3: Neural Network - Manual vs PyTorch

Now the bigger rebuild. Part 2's neural network had 10 parameters, a wiggle method for gradients, and a full backpropagation function.

First, let's run the manual version for comparison (abbreviated from Part 2).

In [ ]:
import random

# Curved data from Part 2
sqft2  = [400,  600,  800, 1000, 1200, 1500, 1800, 2200, 2800, 3500]
price2 = [ 80,  150,  195,  250,  290,  350,  430,  550,  720,  980]

# Normalized
sqft2_norm  = [s / 1000.0 for s in sqft2]
price2_norm = [p / 100.0 for p in price2]

def relu(x):
    return max(0.0, x)

class ManualNetwork:
    def __init__(self):
        random.seed(42)
        self.hw = [random.uniform(-1, 1) for _ in range(3)]
        self.hb = [random.uniform(-1, 1) for _ in range(3)]
        self.ow = [random.uniform(-1, 1) for _ in range(3)]
        self.ob = random.uniform(-1, 1)

    def forward(self, x):
        self.hidden = [relu(self.hw[i] * x + self.hb[i]) for i in range(3)]
        return sum(self.ow[i] * self.hidden[i] for i in range(3)) + self.ob

    def get_params(self):
        return self.hw + self.hb + self.ow + [self.ob]

    def set_params(self, params):
        self.hw = params[0:3]
        self.hb = params[3:6]
        self.ow = params[6:9]
        self.ob = params[9]

def manual_loss(net):
    total = 0
    for x, actual in zip(sqft2_norm, price2_norm):
        total += (net.forward(x) - actual) ** 2
    return total / len(sqft2_norm)

def manual_backprop(net):
    n = len(sqft2_norm)
    total = [0.0] * 10
    for x, actual in zip(sqft2_norm, price2_norm):
        h_raw = [net.hw[i] * x + net.hb[i] for i in range(3)]
        h_out = [relu(h_raw[i]) for i in range(3)]
        pred = sum(net.ow[i] * h_out[i] for i in range(3)) + net.ob
        d = 2 * (pred - actual)
        d_ow = [d * h_out[i] for i in range(3)]
        d_ob = d
        d_h = [d * net.ow[i] * (1 if h_raw[i] > 0 else 0) for i in range(3)]
        d_hw = [d_h[i] * x for i in range(3)]
        d_hb = [d_h[i] for i in range(3)]
        grads = d_hw + d_hb + d_ow + [d_ob]
        for i in range(10):
            total[i] += grads[i] / n
    return total

# Train manual network
net_manual = ManualNetwork()
for epoch in range(2000):
    grads = manual_backprop(net_manual)
    params = net_manual.get_params()
    for i in range(10):
        params[i] -= 0.01 * grads[i]
    net_manual.set_params(params)
    if epoch % 400 == 0:
        print(f"Epoch {epoch:4d} | Loss: {manual_loss(net_manual):.4f}")

print(f"\nFinal loss: {manual_loss(net_manual):.4f}")

### Now the PyTorch version

Same architecture: 1 input -> 3 hidden neurons with ReLU -> 1 output.

In [ ]:
class PyTorchNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(1, 3)   # 3 hidden neurons
        self.relu = nn.ReLU()            # same activation from Part 2
        self.output = nn.Linear(3, 1)    # 1 output neuron

    def forward(self, x):
        x = self.relu(self.hidden(x))
        return self.output(x)

# Same curved data as tensors
X2 = torch.tensor([[400],[600],[800],[1000],[1200],
                   [1500],[1800],[2200],[2800],[3500]],
                  dtype=torch.float32)
y2 = torch.tensor([[80],[150],[195],[250],[290],
                   [350],[430],[550],[720],[980]],
                  dtype=torch.float32)

# Normalize (same as Part 2)
X2_norm = X2 / 1000.0
y2_norm = y2 / 1000.0

torch.manual_seed(42)
net_torch = PyTorchNetwork()
criterion = nn.MSELoss()
optimizer = optim.SGD(net_torch.parameters(), lr=0.01)

# Training loop - same 3 lines as the linear model
for epoch in range(2000):
    predictions = net_torch(X2_norm)
    loss = criterion(predictions, y2_norm)

    optimizer.zero_grad()
    loss.backward()          # autograd handles backprop!
    optimizer.step()

    if epoch % 400 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss.item():.4f}")

print(f"\nFinal loss: {loss.item():.4f}")
print(f"Parameters: {sum(p.numel() for p in net_torch.parameters())}")

In [ ]:
# Compare predictions
print("Sqft  | Actual | Manual  | PyTorch | Off (Manual) | Off (PyTorch)")
print("------|--------|---------|---------|--------------|-------------")

net_torch.eval()
with torch.no_grad():
    for s, p, x_n, y_n in zip(sqft2, price2, sqft2_norm, y2_norm):
        pred_m = net_manual.forward(x_n) * 100
        pred_t = net_torch(torch.tensor([[x_n]])).item() * 1000
        print(f"{s:5d} | ${p:5d}k | ${pred_m:5.0f}k | ${pred_t:5.0f}k | "
              f"${abs(p - pred_m):10.1f}k | ${abs(p - pred_t):10.1f}k")

print("\nBoth networks follow the curve. Same architecture, same results.")

---
## Part 4: What We Eliminated

Here's what PyTorch replaced:

| Component | Manual (Part 2) | PyTorch |
|-----------|----------------|---------|
| Model definition | 12 lines | 7 lines |
| Forward pass | 3 lines | 2 lines |
| Gradient computation | 15 lines (wiggle) or 12 lines (backprop) | 1 line |
| Parameter update | 4 lines | 1 line |
| **Total** | **~34 lines** | **~11 lines** |

Two-thirds of the code was gradient plumbing. PyTorch erased it.

**What changed:** gradient computation, parameter management, optimization.

**What didn't change:** the training loop (predict, measure, adjust, repeat), the architecture, the math, the results.

The framework does the calculus. You still design the architecture.

---
## Part 5: Visualize Both Models

In [ ]:
def ascii_compare(sqft_data, price_data, predict_fn, label="model"):
    """ASCII plot showing data points vs model predictions."""
    width, height = 55, 18
    min_x, max_x = 200, 3800
    min_y, max_y = 30, 1050
    grid = [[' ']*width for _ in range(height)]

    def to_grid(x, y):
        c = int((x - min_x) / (max_x - min_x) * (width - 1))
        r = int((1 - (y - min_y) / (max_y - min_y)) * (height - 1))
        return max(0, min(height-1, r)), max(0, min(width-1, c))

    for px in range(width):
        x = min_x + px / (width-1) * (max_x - min_x)
        y = predict_fn(x)
        if min_y <= y <= max_y:
            r, c = to_grid(x, y)
            grid[r][c] = '-'

    for x, y in zip(sqft_data, price_data):
        r, c = to_grid(x, y)
        grid[r][c] = '*'

    for row in grid:
        print(f"  |{''.join(row)}|")
    print(f"  * = actual    - = {label}")

# PyTorch neural network predictions
def torch_predict(x_raw):
    with torch.no_grad():
        return net_torch(torch.tensor([[x_raw / 1000.0]])).item() * 1000

print("PyTorch neural network vs actual data:\n")
ascii_compare(sqft2, price2, torch_predict, "PyTorch network")

---
## Part 6: Predict New Houses

In [ ]:
new_houses = [500, 900, 1300, 2000, 2500, 3000, 4000]

print("Predictions for houses neither model has seen:\n")
print("Sqft  | Manual  | PyTorch")
print("------|---------|--------")

with torch.no_grad():
    for s in new_houses:
        pred_m = net_manual.forward(s / 1000) * 100
        pred_t = net_torch(torch.tensor([[s / 1000.0]])).item() * 1000
        print(f"{s:5d} | ${pred_m:5.0f}k | ${pred_t:5.0f}k")

print("\nBoth networks generalize to unseen data.")

---
## Try It Yourself

Experiments to try:

1. **Swap the optimizer:** Change `optim.SGD` to `optim.Adam(model.parameters(), lr=0.001)` - does it converge faster?

2. **More neurons:** Change `nn.Linear(1, 3)` to `nn.Linear(1, 10)` (and `nn.Linear(10, 1)` for output) - does the curve fit better?

3. **Different activation:** Replace `nn.ReLU()` with `nn.Tanh()` - how does the curve shape change?

4. **Two hidden layers:** Add a second layer to the network:
```python
class DeeperNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
        )
    def forward(self, x):
        return self.net(x)
```

5. **Remove ReLU:** What happens if you delete the activation function entirely? Does the network collapse to a straight line?